# Pre-SFT Baseline: Context-Parametric Inversion

Establishes the true step-0 anchor for the CPI trajectory: evaluates the raw
pretrained base model (no LoRA adapter -- the exact pre-SFT state both the
`alpaca` and `tulu` runs started from) on the conflict-eval set.

Two passes:
1. **Logprob-only** (`eval.py --logprob-only`): Layer 0 (filter) + Layer 1
   (`method_logprob`) are both teacher-forced log-prob comparisons over the
   candidate answers, so neither needs the model to generate free-form
   text -- the right first pass for a raw base model that can't reliably
   follow the "answer in a few words" instruction.
2. **Judge-escalated** (`eval.py --use-judge`, default v2 prompt): for
   whatever Layer 1 leaves AMBIG (a large fraction for a never-fine-tuned
   base model), generates a free-form continuation and lets the judge read
   it -- even a rambling/unpolished continuation can carry enough signal
   for the judge to call CTX/PAR/OTHER. This uses `--judge-prompt-version
   v2` (the default), never v1 -- v1 has a known systematic PAR-bias, see
   `docs/label-audit-findings.md` root cause #1 (only relevant if you're
   reading this file from an older checkout that predates the CLI default).

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Get the codebase (`dev` branch)

Clones fresh if not already present; otherwise fetches and fast-forwards to
the latest `dev`. `dev` is where active work lands (the judge-fix-only
correction + this baseline's `--logprob-only`/`--judge-prompt-version` flags
are all already there).

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!git log --oneline -3
print("\nRepo dir:", REPO_DIR)

## 2. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub openai

## 3. Configuration -- checkpoint locations on Drive + secrets

`CPI_CKPT_DIR_ALPACA` / `CPI_CKPT_DIR_TULU` point at the same Drive folders
used elsewhere (each directly contains `checkpoint-<step>/` and `final/`) --
not needed for the baseline eval itself (no adapter is loaded), but recorded
here so this notebook is self-contained for whatever comes after the
baseline run.

Secrets are pulled from Colab's secrets manager (the key icon in the left
sidebar) via `userdata.get(...)`, never pasted inline -- this notebook lives
under `experiment-notebooks/` and IS tracked by git, unlike `src/runs.ipynb`.

**IMPORTANT**: if you ever see this cell (or any cell) come back from Colab
with a real token/key hardcoded in place of a `userdata.get(...)` call --
that happens if Colab's autosave writes the executed notebook back over
this file -- do NOT commit it. Revert the cell to `userdata.get(...)` and
rotate the exposed credential.

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

# <-- EDIT if your Drive paths differ
os.environ["CPI_CKPT_DIR_ALPACA"] = "/content/drive/MyDrive/Checkpoints - Alpaca - CPI- Analysis/Alpaca_Checkpoints/checkpoints"
os.environ["CPI_CKPT_DIR_TULU"]   = "/content/drive/MyDrive/Checkpoints - TULU - CPI- Analysis/checkpoints"

# Base model both runs actually trained on -- congif.yaml's run.model / models.<name>.hf_id
BASE_MODEL = "meta-llama/Llama-2-7b-hf"  # <-- EDIT if different

# Add HF_TOKEN and OPENAI_API_KEY as Colab secrets first (key icon, left
# sidebar) -- HF_TOKEN for the gated Llama-2 weights, OPENAI_API_KEY for the
# Layer-2 judge (only needed for section 4b below).
login(token=userdata.get("HF_TOKEN"))
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("CPI_CKPT_DIR_ALPACA:", os.environ["CPI_CKPT_DIR_ALPACA"])
print("CPI_CKPT_DIR_TULU  :", os.environ["CPI_CKPT_DIR_TULU"])
print("BASE_MODEL         :", BASE_MODEL)

## 4. Run the pre-SFT baseline eval (Layers 0+1 only, no generation)

Loads the raw pretrained base model -- no adapter applied, i.e. the true
step-0 state -- with `use_chat_template=True` (default), matching every
existing checkpoint's recorded `config`. Writes
`results/cpi-results/presft-baseline/Llama-2-7b-hf_eval.json` in the same
schema as `results/phase0-results/model_*/results.json`.

In [ ]:
!python src/evaluation/eval.py \
    --model-id {BASE_MODEL} \
    --dataset dataset/conflict_eval_unified.json \
    --output-dir results/cpi-results/presft-baseline \
    --logprob-only

## 4b. Judge-escalate the AMBIG items (needs OPENAI_API_KEY, section 3 above)

Re-runs WITHOUT `--logprob-only` -- this generates a free-form continuation
per item (paper/ordered diagnostics get populated as a side effect, but
aren't load-bearing) and escalates every Layer-1 AMBIG item to the judge.
`--judge-prompt-version v2` is the CLI default already; passed explicitly
here as a reminder never to use v1 for a new run. Writes to a SEPARATE
output dir so it doesn't overwrite section 4's logprob-only result.

In [ ]:
!python src/evaluation/eval.py \
    --model-id {BASE_MODEL} \
    --dataset dataset/conflict_eval_unified.json \
    --output-dir results/cpi-results/presft-baseline-judged \
    --use-judge \
    --judge-prompt-version v2

## 5. Copy the results to Drive (optional, keeps them if the Colab runtime resets)

In [ ]:
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/cpi-presft-baseline"  # <-- EDIT if you want a different Drive spot
!mkdir -p "{DRIVE_RESULTS_DIR}/logprob-only" "{DRIVE_RESULTS_DIR}/judged"
!cp -v results/cpi-results/presft-baseline/*.json "{DRIVE_RESULTS_DIR}/logprob-only/"
!cp -v results/cpi-results/presft-baseline-judged/*.json "{DRIVE_RESULTS_DIR}/judged/"